# Skin Cancer Detection - Multimodal (image + patient metadata)

EfficientNetB0 image branch + metadata branch (age, sex, lesion location),
fused for a 7-class prediction. Goal: clear the high-80s and push toward ~90%+.

**Runtime:** Runtime -> Change runtime type -> GPU (T4).

> Educational demo only. NOT a medical device. Do not use for diagnosis.

Class order is alphabetical and matches `CLASS_ORDER` in `backend/main.py`.
The metadata encoding constants MUST match the backend exactly.

# 1. Download HAM10000 from Kaggle

In [ ]:
import os, json

# Credentials set directly (no kaggle.json upload needed).
# If auth fails (401/403), try the key WITHOUT the 'KGAT_' prefix.
KAGGLE_USERNAME = 'imtiyazakiwat'
KAGGLE_KEY = 'KGAT_9c0be8a0ebade8314cf3a6b7d7407da7'

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

!pip -q install kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham --unzip
print('Downloaded. Files:')
!ls /content/ham

# 2. Import libraries + GPU check

In [ ]:
import glob, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE = 224

gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'No GPU! Set Runtime -> Change runtime type -> T4 GPU, then re-run.'

# 3. Build the dataframe (image path + metadata)

In [ ]:
BASE = '/content/ham'
meta = pd.read_csv(f'{BASE}/HAM10000_metadata.csv')

paths = {os.path.splitext(os.path.basename(p))[0]: p
         for p in glob.glob(f'{BASE}/**/*.jpg', recursive=True)}
meta['path'] = meta['image_id'].map(paths)
meta = meta.dropna(subset=['path']).reset_index(drop=True)

meta['age'] = meta['age'].fillna(meta['age'].median())
meta['sex'] = meta['sex'].fillna('unknown').str.lower()
meta['localization'] = meta['localization'].fillna('unknown').str.lower()

print(meta[['dx', 'age', 'sex', 'localization']].head())
print('\nTotal images:', len(meta))

# 4. Images per class

In [ ]:
counts = meta['dx'].value_counts().sort_index()
print(counts)
plt.figure(figsize=(9, 5))
sns.barplot(x=counts.index, y=counts.values, palette='viridis')
plt.title('Images per class (HAM10000 - imbalanced)')
plt.ylabel('count'); plt.xlabel('class'); plt.show()

# 5. One sample image per class

In [ ]:
labels = sorted(meta['dx'].unique())
fig, ax = plt.subplots(1, len(labels), figsize=(18, 4))
for i, c in enumerate(labels):
    row = meta[meta['dx'] == c].iloc[0]
    img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
    ax[i].imshow(img); ax[i].set_title(c); ax[i].axis('off')
plt.tight_layout(); plt.show()
print('labels (class order):', labels)

# 6. Metadata encoding + stratified split (no leakage)

Constants MUST match `backend/main.py`. Metadata vector =
`[age/100] + one-hot(sex) + one-hot(localization)` = 19 numbers.

In [ ]:
SEX_CATEGORIES = ['female', 'male', 'unknown']
LOC_CATEGORIES = ['abdomen', 'acral', 'back', 'chest', 'ear', 'face',
                  'foot', 'genital', 'hand', 'lower extremity', 'neck',
                  'scalp', 'trunk', 'unknown', 'upper extremity']
AGE_FILL = 50.0
META_DIM = 1 + len(SEX_CATEGORIES) + len(LOC_CATEGORIES)  # 19
label_to_idx = {c: i for i, c in enumerate(labels)}

def encode_meta(age, sex, loc):
    out = []
    try:
        a = float(age)
        if np.isnan(a):
            a = AGE_FILL
    except (TypeError, ValueError):
        a = AGE_FILL
    out.append(min(max(a, 0.0), 100.0) / 100.0)
    s = str(sex).lower().strip()
    s = s if s in SEX_CATEGORIES else 'unknown'
    out += [1.0 if s == c else 0.0 for c in SEX_CATEGORIES]
    l = str(loc).lower().strip()
    l = l if l in LOC_CATEGORIES else 'unknown'
    out += [1.0 if l == c else 0.0 for c in LOC_CATEGORIES]
    return out

train_df, temp_df = train_test_split(
    meta, test_size=0.30, stratify=meta['dx'], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['dx'], random_state=SEED)
print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))

# 7. Class weights + in-RAM multimodal generator

Images are decoded into memory ONCE (much faster than re-reading from disk
every epoch). Augmentation is applied per-batch to the training split only.
Class weights (passed as sample weights) handle the imbalance.

In [ ]:
classes = np.array(labels)
weights = compute_class_weight('balanced', classes=classes,
                               y=train_df['dx'].values)
class_weight = {label_to_idx[c]: w for c, w in zip(classes, weights)}
print('class_weight:', class_weight)

class MultiModalSeq(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size=32, training=False):
        self.bs = batch_size
        self.training = training
        self.aug = ImageDataGenerator(
            rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
            zoom_range=0.15, horizontal_flip=True, vertical_flip=True,
            fill_mode='reflect')
        imgs, metas, ys = [], [], []
        desc = 'train' if training else 'eval'
        for _, r in tqdm(df.iterrows(), total=len(df), desc=f'load {desc}'):
            img = cv2.cvtColor(cv2.imread(r['path']), cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            imgs.append(img)
            metas.append(encode_meta(r['age'], r['sex'], r['localization']))
            ys.append(label_to_idx[r['dx']])
        self.images = np.array(imgs, dtype=np.uint8)
        self.metas = np.array(metas, dtype=np.float32)
        self.labels = np.array(ys, dtype=np.int32)
        self.sw = np.array([class_weight[y] for y in ys], dtype=np.float32)
        self.idx = np.arange(len(self.images))
        if training:
            np.random.shuffle(self.idx)

    def __len__(self):
        return int(np.ceil(len(self.images) / self.bs))

    def __getitem__(self, i):
        ids = self.idx[i * self.bs:(i + 1) * self.bs]
        imgs = self.images[ids].astype('float32')
        if self.training:
            imgs = np.stack([self.aug.random_transform(im) for im in imgs])
        imgs = imgs / 255.0
        y = tf.keras.utils.to_categorical(self.labels[ids], num_classes=len(labels))
        return (imgs, self.metas[ids]), y, self.sw[ids]

    def on_epoch_end(self):
        if self.training:
            np.random.shuffle(self.idx)

train_gen = MultiModalSeq(train_df, 32, training=True)
val_gen = MultiModalSeq(val_df, 32, training=False)
test_gen = MultiModalSeq(test_df, 32, training=False)

# 8. Build the multimodal model

Image input is [0,1] (matching the backend, which divides by 255); a
Rescaling layer converts to the [0,255] range EfficientNet expects.

In [ ]:
def build_model():
    img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
    x = layers.Rescaling(255.0)(img_in)  # [0,1] -> [0,255] for EfficientNet
    base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)
    base.trainable = False
    img_feat = layers.GlobalAveragePooling2D()(base.output)
    img_feat = layers.Dropout(0.3)(img_feat)

    meta_in = layers.Input(shape=(META_DIM,), name='metadata')
    m = layers.Dense(32, activation='relu')(meta_in)
    m = layers.Dropout(0.3)(m)
    m = layers.Dense(16, activation='relu')(m)

    z = layers.Concatenate()([img_feat, m])
    z = layers.Dense(128, activation='relu')(z)
    z = layers.Dropout(0.4)(z)
    out = layers.Dense(len(labels), activation='softmax')(z)
    return Model([img_in, meta_in], out), base

model, base = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
print('trainable params (head phase):',
      int(np.sum([np.prod(v.shape) for v in model.trainable_weights])))

# 9. Stage 1 - train the head (EfficientNet frozen)

Note: fresh callbacks are created for THIS stage only (reusing callback
objects across stages corrupts early-stopping/checkpoint state).

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

cb_head = [
    ModelCheckpoint('best_head.keras', monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1),
]
hist1 = model.fit(train_gen, validation_data=val_gen, epochs=12, callbacks=cb_head)

# 10. Stage 2 - fine-tune EfficientNet (BatchNorm kept frozen)

We unfreeze the top layers but KEEP every BatchNormalization layer frozen.
Unfreezing BatchNorm is the classic cause of EfficientNet fine-tuning
collapse, freezing it keeps the pretrained statistics stable.

In [ ]:
base.trainable = True
for layer in base.layers[:-60]:      # only top ~60 layers train
    layer.trainable = False
for layer in base.layers:            # BatchNorm ALWAYS frozen
    if isinstance(layer, BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])
print('trainable params (fine-tune):',
      int(np.sum([np.prod(v.shape) for v in model.trainable_weights])))

cb_ft = [
    ModelCheckpoint('best_finetune.keras', monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]
hist2 = model.fit(train_gen, validation_data=val_gen, epochs=18, callbacks=cb_ft)

# 11. Keep whichever stage produced the best validation accuracy

Safety net: if fine-tuning didn't help, we fall back to the head model so we
never end up worse than where we started.

In [ ]:
head_model = tf.keras.models.load_model('best_head.keras')
ft_model = tf.keras.models.load_model('best_finetune.keras')

head_acc = head_model.evaluate(val_gen, verbose=0)[1]
ft_acc = ft_model.evaluate(val_gen, verbose=0)[1]
print(f'head val acc: {head_acc:.4f}  |  fine-tune val acc: {ft_acc:.4f}')

best_model = ft_model if ft_acc >= head_acc else head_model
print('Using:', 'fine-tuned' if ft_acc >= head_acc else 'head-only', 'model')

# 12. Evaluate the best model on the held-out test set

In [ ]:
loss, acc = best_model.evaluate(test_gen)
print(f'\nTEST ACCURACY: {acc:.4f}  |  TEST LOSS: {loss:.4f}')

In [ ]:
y_true, y_pred = [], []
for (Xi, Xm), y, _ in test_gen:
    p = best_model.predict([Xi, Xm], verbose=0)
    y_pred.extend(np.argmax(p, axis=1))
    y_true.extend(np.argmax(y, axis=1))
print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels, linewidths=1, linecolor='black')
plt.title('Confusion matrix'); plt.xlabel('predicted'); plt.ylabel('true'); plt.show()

# 13. Training curves

In [ ]:
acc = hist1.history['accuracy'] + hist2.history['accuracy']
val = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
plt.figure(figsize=(9, 5))
plt.plot(acc, label='train acc')
plt.plot(val, label='val acc')
plt.axvline(len(hist1.history['accuracy']) - 1, color='gray', linestyle='--', label='fine-tune start')
plt.title('Accuracy'); plt.xlabel('epoch'); plt.legend(); plt.show()

# 14. Export the BEST model for the backend

Exports the selected best model (not whatever happened to be in memory).
Download the zip, then put **model.keras** into `backend/model/`.

In [ ]:
os.makedirs('/content/export', exist_ok=True)
best_model.save('/content/export/model.keras')

schema = {
    'class_order': labels,
    'sex_categories': SEX_CATEGORIES,
    'loc_categories': LOC_CATEGORIES,
    'age_fill': AGE_FILL,
    'meta_dim': META_DIM,
    'img_size': IMG_SIZE,
}
with open('/content/export/metadata_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)

!cd /content/export && zip -r /content/skin_model_export.zip .
from google.colab import files
files.download('/content/skin_model_export.zip')
print('Done. Put model.keras into backend/model/.')